In [0]:
import re
from pyspark.sql import functions as F

storage_account = "group3gpsa"

source_dir = f"abfss://landing@{storage_account}.dfs.core.windows.net/MonthlyNycData"
target_dir = f"abfss://landing@{storage_account}.dfs.core.windows.net/CombinedYearlyParquet"

type_labels = {
    "yellow": "Yellow",
    "green": "Green",
    "fhv": "For Hire Vehicles",
    "fhvhv": "High Volume FHV"
}

#lägg till de år man vill kolla (20xx|20xx osv)
pattern = re.compile(
    r"^(yellow|green|fhv|fhvhv)_tripdata_(2011|2012|2013|2014|2015|2016|2017|2018|2019|2020|2021|2022|2023|2024|2025|2026)-\d{2}\.parquet$",
    re.IGNORECASE
)

all_items = dbutils.fs.ls(source_dir)

source_files = {}
for f in all_items:
    if f.isDir():
        continue
    m = pattern.match(f.name)
    if m:
        trip_type = m.group(1).lower()
        year = m.group(2)
        key = (trip_type, year)
        if key not in source_files:
            source_files[key] = []
        source_files[key].append(f.path)

print(f"Found {len(source_files)} type/year groups")

for (trip_type, year), paths in sorted(source_files.items()):
    paths = sorted(paths)

    final_name = f"{trip_type}_tripdata_{year}-00.parquet"
    final_parquet = f"{target_dir}/{final_name}"
    temp_dir = f"{target_dir}/_tmp_{trip_type}_{year}"

    print(f"Starting: {trip_type}, {year}")
    print(f"Source files: {len(paths)}")
    print(f"Target:       {final_parquet}")

    try:
        dbutils.fs.rm(temp_dir, True)
    except:
        pass

    try:
        dbutils.fs.rm(final_parquet, True)
    except:
        pass

    combined_df = None

    for p in paths:
        print(f"Reading: {p}")

        df = spark.read.parquet(p)

        #force every column in this file to string
        df = df.select([F.col(c).cast("string").alias(c) for c in df.columns])

        if combined_df is None:
            combined_df = df
        else:
            combined_df = combined_df.unionByName(df, allowMissingColumns=True)

    (
        combined_df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(temp_dir)
    )

    written_files = dbutils.fs.ls(temp_dir)
    part_file = [f.path for f in written_files if f.name.startswith("part-") and f.name.endswith(".parquet")][0]

    dbutils.fs.mv(part_file, final_parquet)
    dbutils.fs.rm(temp_dir, True)

    print(f"Finished: {final_parquet}")

print("All done.")